
# <font color="green">Floating-point comparison (max of two doubles)</font>

## Problem

* Write a function `dmax` that returns the larger of two `double` values, __in assembly__.
* You may use either a conditional branch or a conditional select (`fcsel`); the point is to compare the two floating-point numbers with `fcmp`/`fcmpe`.
* That is, translate the following C function into assembly:
```
double dmax(double a, double b) {
  if (a > b) { return a; } else { return b; }
}
```
* Fill in the skeleton `dmax.s` (after `// ------- write your answer here -------`).
* The checker `check_dmax.c` verifies your result. If you see `OK`s and no errors, you are done.

## Hints

* Comparing floating-point numbers uses different instructions from integer comparison. You do not need to memorize them --- let `gcc -S` show you the instruction name and look it up.
* The *Observe* cells below contain a simple example, `cmp_sign` (returns `1.0` if `a > b`, else `-1.0`). Compile it and observe that `fcmp`/`fcmpe` is the comparison instruction (the floating-point counterpart of `cmp`), followed by a conditional select (`fcsel`). This problem uses the same comparison, but selects between `a` and `b` themselves.
* Note: if you write a plain `a > b ? a : b` in C, the compiler folds it into a single `fmaxnm` instruction; that is why the example above returns two distinct constants, to keep the `fcmp` + `fcsel` pattern visible. For your hand-written answer, an explicit `fcmp` + `fcsel` (or a branch) is perfectly fine.



# 1. AI Tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback.
* Execute the following cell before you use it.

In [ ]:
import heytutor

## 1-2. Examples
* A general question
```
%%hey
What does the `ldr` instruction do in ARM64?
```

* A hint on this specific problem
```
%%hey problem_file=dmax.md
Give me a hint on this problem.

{problem}
```

* Builtin variables usable in `%%hey` cells
  * `{file:FILENAME}` is the content of FILE
  * `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` the second last, etc.
  * `{problem}` is the content of the file you specify by `%%hey problem_file=foo.md`
  * `{answer}` is the content of the file you specify by `%%hey answer_file=foo.s`


# 2. Observe: compile example functions
* Before writing your own assembly, it helps to see what the compiler generates for related example functions.
* Running the first cell below writes `explore.c` (some small example functions related to this problem).
* The second cell compiles it with `gcc -O3 -S` and prints the generated assembly.
* Feel free to edit `explore.c` (change the code, add functions, change constants) and re-run the two cells to see how the assembly changes.

In [ ]:
%%writefile_ explore.c
/* Comparing two floating-point numbers uses fcmp / fcmpe (not the integer cmp).
   This simple example returns 1.0 if a > b, else -1.0. Observe fcmpe followed by
   a conditional select (fcsel) choosing between the two candidate results.
   (A plain max would fold into a single fmaxnm, hiding the comparison, so this
   example returns two distinct constants instead.) */
double cmp_sign(double a, double b) {
  return a > b ? 1.0 : -1.0;
}

In [ ]:
%%bash_
gcc -O3 -S explore.c
cat explore.s


# 3. Your Answer (assembly)
* Running the cell below writes the skeleton assembly file `dmax.s`.
* Fill in your instructions after the line `// ------- write your answer here -------`, then run the cell again to save it.

In [ ]:
%%writefile_ dmax.s
	.arch armv8-a
	.file	"dmax.c"
	.text
	.align	2
	.p2align 4,,11
	.global	dmax
	.type	dmax, %function
dmax:
.LFB0:
	.cfi_startproc
	// ------- write your answer here -------
	.cfi_endproc
.LFE0:
	.size	dmax, .-dmax
	.ident	"GCC: (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0"
	.section	.note.GNU-stack,"",@progbits


# 4. Checker
* The following C program calls your `dmax` function and checks the result against a reference computed in C.

In [ ]:
%%writefile_ check_dmax.c
#include <assert.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
double dmax(double a, double b);
int main(int argc, char ** argv) {
  assert(argc == 3);
  double a = atof(argv[1]);
  double b = atof(argv[2]);
  double r = dmax(a, b);
  double rc = (a > b) ? a : b;
  if (fabs(r - rc) <= 1e-9 * (1.0 + fabs(rc))) { printf("OK %f %f\n", r, rc); return 0; }
  else { printf("NG %f %f\n", r, rc); return 1; }
}


# 5. Compile
* Compile your assembly together with the checker.
* If you get an error, fix `dmax.s` above and recompile.

In [ ]:
%%bash_
gcc -o check_dmax -O3 check_dmax.c dmax.s -lm


# 6. Run
* The commands to run the checker are stored in `run.sh`.
* If you see `OK`s and no errors, you are done.

In [ ]:
%%bash_
./check_dmax 1.5 2.5
./check_dmax 2.5 1.5
./check_dmax -1.0 -3.0


# 7. If things do not go well
* If your program compiles but does not produce the correct answer, run it within a debugger (gdb).
* Compile with `-O0 -g` first:
```
gcc -o check_dmax -O0 -g check_dmax.c dmax.s -lm
```
* Then, in a terminal (SSH or Jupyter terminal):
```
gdb check_dmax
(gdb) break dmax
(gdb) run ...        # give the same arguments as in run.sh
```
* Step through one instruction at a time with `step`, and inspect registers with `print $x0` or `info registers`.

# 8. Ask Questions or Get Feedback
* You are encouraged to ask for feedback once you think you are done, to know if there is a better answer.

In [ ]:
%%hey problem_file=dmax.md answer_file=dmax.s

Problem:
{problem}

My Answer:
{answer}

Give me a feedback to my answer.